# 20 HistGradientBoosting Optuna Prep for J_D

## Import

In [2]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import joblib

import matplotlib as mpl
import matplotlib.pyplot as plt

In [3]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [4]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [5]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
full     [595212, 595212, 595212]
dtype: object

## Hilfsvariablen

In [6]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [7]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

## Preproc nach J_D

In [8]:
targ_enc = TargetEncoder(
    cv=5,
    shuffle=True,
    random_state=RANDOM_STATE,
    smooth=50.0
)

In [9]:
te_train = pd.DataFrame(
    targ_enc.fit_transform(x_train[cat_cols], y_train),
    columns=cat_cols,
    index=x_train.index
)

te_val = pd.DataFrame(
    targ_enc.transform(x_val[cat_cols]),
    columns=cat_cols,
    index=x_val.index
)

te_test = pd.DataFrame(
    targ_enc.transform(x_test[cat_cols]),
    columns=cat_cols,
    index=x_test.index
)

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


In [10]:
x_train_hgb = x_train.copy()
x_val_hgb = x_val.copy()
x_test_hgb = x_test.copy()

x_train_hgb[cat_cols] = te_train
x_val_hgb[cat_cols] = te_val
x_test_hgb[cat_cols] = te_test

In [11]:
x_train_hgb_no_calc = x_train_hgb[feat_cols_no_calc]
x_val_hgb_no_calc = x_val_hgb[feat_cols_no_calc]
x_test_hgb_no_calc = x_test_hgb[feat_cols_no_calc]

In [12]:
pd.Series({
    "train": [len(x_train), len(x_train_hgb), len(x_train_hgb_no_calc)],
    "val": [len(x_val), len(x_val_hgb), len(x_val_hgb_no_calc)],
    "test": [len(x_test), len(x_test_hgb), len(x_test_hgb_no_calc)]
})

train    [476168, 476168, 476168]
val         [59522, 59522, 59522]
test        [59522, 59522, 59522]
dtype: object

## Optuna Hilfe

## Suchräume

|Parameter|Bereich|
|---|---|
|learning_rate|0.003-0.3 (log)|
|min_samples_leaf|5-40000 (log)|
|l2_regularization|1e-8 - 1e5 (log)|
|max_features|0.05-0.35|
|max_bins|104-255|
|interaction_cst|None / pairwise / no_interactions|
|validation_fraction|0.01-0.1|
|n_iter_no_change|5-40|
|tol|1e-10 - 1e-5 (log)|

Für händische Versuche entweder:
|Parameter|Bereich|
|---|---|
|max_leaf_nodes|2-63|
|max_depth|1-6|

In [13]:
fixed_params_leaf = {
    "random_state": RANDOM_STATE,
    "verbose": 0,
    "loss": "log_loss",
    "class_weight": None,
    "categorical_features": None,
    "scoring": "loss",
    "max_iter": 5000,
    "early_stopping": True,
    "max_depth": None
}

In [14]:
fixed_params_depth = {
    "random_state": RANDOM_STATE,
    "verbose": 0,
    "loss": "log_loss",
    "class_weight": None,
    "categorical_features": None,
    "scoring": "loss",
    "max_iter": 5000,
    "early_stopping": True,
    "max_leaf_nodes": None
}

In [15]:
def suchraum_params_leaf(trial):
    """Suchraum definition, leaf"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 40000, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 1e5, log=True),
        "max_features": trial.suggest_float("max_features", 0.05, 0.35),
        "max_bins": trial.suggest_int("max_bins", 104, 255),
        "interaction_cst": trial.suggest_categorical("interaction_cst", [None, "pairwise", "no_interactions"]),
        "validation_fraction": trial.suggest_float("validation_fraction", 0.01, 0.1),
        "n_iter_no_change": trial.suggest_int("n_iter_no_change", 5, 40),
        "tol": trial.suggest_float("tol", 1e-10, 1e-5, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 63)
    }

    return params

In [16]:
def suchraum_params_depth(trial):
    """Suchraum definition, depth"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 40000, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 1e5, log=True),
        "max_features": trial.suggest_float("max_features", 0.05, 0.35),
        "max_bins": trial.suggest_int("max_bins", 104, 255),
        "interaction_cst": trial.suggest_categorical("interaction_cst", [None, "pairwise", "no_interactions"]),
        "validation_fraction": trial.suggest_float("validation_fraction", 0.01, 0.1),
        "n_iter_no_change": trial.suggest_int("n_iter_no_change", 5, 40),
        "tol": trial.suggest_float("tol", 1e-10, 1e-5, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 6)
    }

    return params

## Leaf mit Calc

In [17]:
def objective_leaf_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_leaf,
        **suchraum_params_leaf(trial)
    )

    modell.fit(x_train_hgb, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb)[:, 1])

In [18]:
study_leaf_calc = optuna.create_study(
    study_name = "HGB_leaf_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

[I 2026-08-19 12:02:59,959] A new study created in RDB with name: HGB_leaf_calc


In [19]:
study_leaf_calc.optimize(
    objective_leaf_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

[I 2026-08-19 12:03:33,751] Trial 0 finished with value: 0.6209433354252369 and parameters: {'learning_rate': 0.01683454924600351, 'min_samples_leaf': 25553, 'l2_regularization': 32.8035800581182, 'max_features': 0.22959754525911097, 'max_bins': 127, 'interaction_cst': 'no_interactions', 'validation_fraction': 0.0641003510568888, 'n_iter_no_change': 30, 'tol': 1.267425589893721e-10, 'max_leaf_nodes': 62}. Best is trial 0 with value: 0.6209433354252369.
[I 2026-08-19 12:03:37,043] Trial 1 finished with value: 0.6296457123327305 and parameters: {'learning_rate': 0.13867767003062487, 'min_samples_leaf': 31, 'l2_regularization': 2.3105989608604262e-06, 'max_features': 0.10502135295603016, 'max_bins': 150, 'interaction_cst': None, 'validation_fraction': 0.06506676052501416, 'n_iter_no_change': 10, 'tol': 2.888838362365319e-09, 'max_leaf_nodes': 24}. Best is trial 1 with value: 0.6296457123327305.
[I 2026-08-19 12:03:51,142] Trial 2 finished with value: 0.6277347833425565 and parameters: {'l

In [20]:
pd.DataFrame({
    "best auc_val": study_leaf_calc.best_value,
    "best gini": 2 * study_leaf_calc.best_value - 1,
    "trials": len(study_leaf_calc.trials),
    "pruned": len([t for t in study_leaf_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_leaf_calc.trials if t.state.name == "FAIL"]),
    "best params": study_leaf_calc.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.6366,0.273201,50,0,0,1.452174e-02
min_samples_leaf,0.6366,0.273201,50,0,0,5.975000e+03
l2_regularization,0.6366,0.273201,50,0,0,1.026213e+01
max_features,0.6366,0.273201,50,0,0,2.477600e-01
max_bins,0.6366,0.273201,50,0,0,2.310000e+02
interaction_cst,0.6366,0.273201,50,0,0,NaN
validation_fraction,0.6366,0.273201,50,0,0,5.731336e-02
n_iter_no_change,0.6366,0.273201,50,0,0,3.800000e+01
tol,0.6366,0.273201,50,0,0,1.209587e-08
max_leaf_nodes,0.6366,0.273201,50,0,0,3.600000e+01


In [21]:
study_leaf_calc.best_params

{'learning_rate': 0.014521736210156313,
 'min_samples_leaf': 5975,
 'l2_regularization': 10.262130162526471,
 'max_features': 0.24775997297590835,
 'max_bins': 231,
 'interaction_cst': None,
 'validation_fraction': 0.0573133556070173,
 'n_iter_no_change': 38,
 'tol': 1.2095870910048285e-08,
 'max_leaf_nodes': 36}

## Leaf ohne Calc

In [22]:
def objective_leaf_no_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_leaf,
        **suchraum_params_leaf(trial)
    )

    modell.fit(x_train_hgb_no_calc, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb_no_calc)[:, 1])

In [23]:
study_leaf_no_calc = optuna.create_study(
    study_name = "HGB_leaf_no_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

[I 2026-08-19 12:41:56,004] A new study created in RDB with name: HGB_leaf_no_calc


In [24]:
study_leaf_no_calc.optimize(
    objective_leaf_no_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

[I 2026-08-19 12:42:33,260] Trial 0 finished with value: 0.6223147167899088 and parameters: {'learning_rate': 0.01683454924600351, 'min_samples_leaf': 25553, 'l2_regularization': 32.8035800581182, 'max_features': 0.22959754525911097, 'max_bins': 127, 'interaction_cst': 'no_interactions', 'validation_fraction': 0.0641003510568888, 'n_iter_no_change': 30, 'tol': 1.267425589893721e-10, 'max_leaf_nodes': 62}. Best is trial 0 with value: 0.6223147167899088.
[I 2026-08-19 12:42:35,179] Trial 1 finished with value: 0.6284335408862172 and parameters: {'learning_rate': 0.13867767003062487, 'min_samples_leaf': 31, 'l2_regularization': 2.3105989608604262e-06, 'max_features': 0.10502135295603016, 'max_bins': 150, 'interaction_cst': None, 'validation_fraction': 0.06506676052501416, 'n_iter_no_change': 10, 'tol': 2.888838362365319e-09, 'max_leaf_nodes': 24}. Best is trial 1 with value: 0.6284335408862172.
[I 2026-08-19 12:42:48,262] Trial 2 finished with value: 0.6290737527817523 and parameters: {'l

In [25]:
pd.DataFrame({
    "best auc_val": study_leaf_no_calc.best_value,
    "best gini": 2 * study_leaf_no_calc.best_value - 1,
    "trials": len(study_leaf_no_calc.trials),
    "pruned": len([t for t in study_leaf_no_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_leaf_no_calc.trials if t.state.name == "FAIL"]),
    "best params": study_leaf_no_calc.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.638318,0.276637,50,0,0,0.004109
min_samples_leaf,0.638318,0.276637,50,0,0,6833.000000
l2_regularization,0.638318,0.276637,50,0,0,52.458189
max_features,0.638318,0.276637,50,0,0,0.296687
max_bins,0.638318,0.276637,50,0,0,202.000000
interaction_cst,0.638318,0.276637,50,0,0,NaN
validation_fraction,0.638318,0.276637,50,0,0,0.013916
n_iter_no_change,0.638318,0.276637,50,0,0,40.000000
tol,0.638318,0.276637,50,0,0,0.000001
max_leaf_nodes,0.638318,0.276637,50,0,0,32.000000


In [26]:
study_leaf_no_calc.best_params

{'learning_rate': 0.004108690179427848,
 'min_samples_leaf': 6833,
 'l2_regularization': 52.45818863330329,
 'max_features': 0.2966869253501312,
 'max_bins': 202,
 'interaction_cst': None,
 'validation_fraction': 0.01391561653087682,
 'n_iter_no_change': 40,
 'tol': 1.4783911269827303e-06,
 'max_leaf_nodes': 32}

## Depth mit Calc

In [27]:
def objective_depth_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_depth,
        **suchraum_params_depth(trial)
    )

    modell.fit(x_train_hgb, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb)[:, 1])

In [28]:
study_depth_calc = optuna.create_study(
    study_name = "HGB_depth_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

[I 2026-08-19 13:07:38,842] A new study created in RDB with name: HGB_depth_calc


In [29]:
study_depth_calc.optimize(
    objective_depth_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

[I 2026-08-19 13:08:09,094] Trial 0 finished with value: 0.6208656496990961 and parameters: {'learning_rate': 0.01683454924600351, 'min_samples_leaf': 25553, 'l2_regularization': 32.8035800581182, 'max_features': 0.22959754525911097, 'max_bins': 127, 'interaction_cst': 'no_interactions', 'validation_fraction': 0.0641003510568888, 'n_iter_no_change': 30, 'tol': 1.267425589893721e-10, 'max_depth': 6}. Best is trial 0 with value: 0.6208656496990961.
[I 2026-08-19 13:08:12,436] Trial 1 finished with value: 0.6265900000833611 and parameters: {'learning_rate': 0.13867767003062487, 'min_samples_leaf': 31, 'l2_regularization': 2.3105989608604262e-06, 'max_features': 0.10502135295603016, 'max_bins': 150, 'interaction_cst': None, 'validation_fraction': 0.06506676052501416, 'n_iter_no_change': 10, 'tol': 2.888838362365319e-09, 'max_depth': 3}. Best is trial 1 with value: 0.6265900000833611.
[I 2026-08-19 13:08:28,265] Trial 2 finished with value: 0.6292753787526822 and parameters: {'learning_rate

In [30]:
pd.DataFrame({
    "best auc_val": study_depth_calc.best_value,
    "best gini": 2 * study_depth_calc.best_value - 1,
    "trials": len(study_depth_calc.trials),
    "pruned": len([t for t in study_depth_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_depth_calc.trials if t.state.name == "FAIL"]),
    "best params": study_depth_calc.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.634704,0.269407,50,0,0,2.158115e-01
min_samples_leaf,0.634704,0.269407,50,0,0,1.850000e+02
l2_regularization,0.634704,0.269407,50,0,0,7.188748e+02
max_features,0.634704,0.269407,50,0,0,2.160953e-01
max_bins,0.634704,0.269407,50,0,0,1.990000e+02
interaction_cst,0.634704,0.269407,50,0,0,NaN
validation_fraction,0.634704,0.269407,50,0,0,9.844242e-02
n_iter_no_change,0.634704,0.269407,50,0,0,1.500000e+01
tol,0.634704,0.269407,50,0,0,2.365868e-10
max_depth,0.634704,0.269407,50,0,0,6.000000e+00


In [31]:
study_depth_calc.best_params

{'learning_rate': 0.2158114975072843,
 'min_samples_leaf': 185,
 'l2_regularization': 718.8748437890337,
 'max_features': 0.2160953488876775,
 'max_bins': 199,
 'interaction_cst': None,
 'validation_fraction': 0.09844241887597069,
 'n_iter_no_change': 15,
 'tol': 2.3658675988459827e-10,
 'max_depth': 6}

## Depth ohne Calc

In [32]:
def objective_depth_no_calc(trial):

    modell = HistGradientBoostingClassifier(
        **fixed_params_depth,
        **suchraum_params_depth(trial)
    )

    modell.fit(x_train_hgb_no_calc, y_train)

    return roc_auc_score(y_val, modell.predict_proba(x_val_hgb_no_calc)[:, 1])

In [33]:
study_depth_no_calc = optuna.create_study(
    study_name = "HGB_depth_no_calc",
    storage = "sqlite:///HGB_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=500)
)

[I 2026-08-19 13:21:19,066] A new study created in RDB with name: HGB_depth_no_calc


In [34]:
study_depth_no_calc.optimize(
    objective_depth_no_calc,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

[I 2026-08-19 13:21:45,955] Trial 0 finished with value: 0.6212898584588418 and parameters: {'learning_rate': 0.01683454924600351, 'min_samples_leaf': 25553, 'l2_regularization': 32.8035800581182, 'max_features': 0.22959754525911097, 'max_bins': 127, 'interaction_cst': 'no_interactions', 'validation_fraction': 0.0641003510568888, 'n_iter_no_change': 30, 'tol': 1.267425589893721e-10, 'max_depth': 6}. Best is trial 0 with value: 0.6212898584588418.
[I 2026-08-19 13:21:49,242] Trial 1 finished with value: 0.6293356607539581 and parameters: {'learning_rate': 0.13867767003062487, 'min_samples_leaf': 31, 'l2_regularization': 2.3105989608604262e-06, 'max_features': 0.10502135295603016, 'max_bins': 150, 'interaction_cst': None, 'validation_fraction': 0.06506676052501416, 'n_iter_no_change': 10, 'tol': 2.888838362365319e-09, 'max_depth': 3}. Best is trial 1 with value: 0.6293356607539581.
[I 2026-08-19 13:22:03,352] Trial 2 finished with value: 0.630655948319442 and parameters: {'learning_rate'

In [35]:
pd.DataFrame({
    "best auc_val": study_depth_no_calc.best_value,
    "best gini": 2 * study_depth_no_calc.best_value - 1,
    "trials": len(study_depth_no_calc.trials),
    "pruned": len([t for t in study_depth_no_calc.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_depth_no_calc.trials if t.state.name == "FAIL"]),
    "best params": study_depth_no_calc.best_params
})

,best auc_val,best gini,trials,pruned,fail,best params
learning_rate,0.634533,0.269067,50,0,0,2.468715e-02
min_samples_leaf,0.634533,0.269067,50,0,0,1.650000e+02
l2_regularization,0.634533,0.269067,50,0,0,4.711124e-03
max_features,0.634533,0.269067,50,0,0,2.469807e-01
max_bins,0.634533,0.269067,50,0,0,1.440000e+02
interaction_cst,0.634533,0.269067,50,0,0,NaN
validation_fraction,0.634533,0.269067,50,0,0,1.482309e-02
n_iter_no_change,0.634533,0.269067,50,0,0,3.400000e+01
tol,0.634533,0.269067,50,0,0,8.370655e-08
max_depth,0.634533,0.269067,50,0,0,6.000000e+00


In [36]:
study_depth_no_calc.best_params

{'learning_rate': 0.024687148414834312,
 'min_samples_leaf': 165,
 'l2_regularization': 0.004711123505991509,
 'max_features': 0.2469806711366302,
 'max_bins': 144,
 'interaction_cst': None,
 'validation_fraction': 0.014823091721617599,
 'n_iter_no_change': 34,
 'tol': 8.37065478390795e-08,
 'max_depth': 6}

## Load Study

In [37]:
study_leaf_calc_loaded = optuna.load_study(
    study_name = "HGB_leaf_calc",
    storage = "sqlite:///HGB_opti.db"
)


In [38]:
study_leaf_no_calc_loaded = optuna.load_study(
    study_name = "HGB_leaf_no_calc",
    storage = "sqlite:///HGB_opti.db"
)

In [39]:
study_depth_calc_loaded = optuna.load_study(
    study_name = "HGB_depth_calc",
    storage = "sqlite:///HGB_opti.db"
)


In [40]:
study_depth_no_calc_loaded = optuna.load_study(
    study_name = "HGB_depth_no_calc",
    storage = "sqlite:///HGB_opti.db"
)

## Best Params again

In [41]:
study_leaf_calc_loaded.best_params

{'learning_rate': 0.014521736210156313,
 'min_samples_leaf': 5975,
 'l2_regularization': 10.262130162526471,
 'max_features': 0.24775997297590835,
 'max_bins': 231,
 'interaction_cst': None,
 'validation_fraction': 0.0573133556070173,
 'n_iter_no_change': 38,
 'tol': 1.2095870910048285e-08,
 'max_leaf_nodes': 36}

In [42]:
study_leaf_no_calc_loaded.best_params

{'learning_rate': 0.004108690179427848,
 'min_samples_leaf': 6833,
 'l2_regularization': 52.45818863330329,
 'max_features': 0.2966869253501312,
 'max_bins': 202,
 'interaction_cst': None,
 'validation_fraction': 0.01391561653087682,
 'n_iter_no_change': 40,
 'tol': 1.4783911269827303e-06,
 'max_leaf_nodes': 32}

In [43]:
study_depth_calc_loaded.best_params

{'learning_rate': 0.2158114975072843,
 'min_samples_leaf': 185,
 'l2_regularization': 718.8748437890337,
 'max_features': 0.2160953488876775,
 'max_bins': 199,
 'interaction_cst': None,
 'validation_fraction': 0.09844241887597069,
 'n_iter_no_change': 15,
 'tol': 2.3658675988459827e-10,
 'max_depth': 6}

In [44]:
study_depth_no_calc_loaded.best_params

{'learning_rate': 0.024687148414834312,
 'min_samples_leaf': 165,
 'l2_regularization': 0.004711123505991509,
 'max_features': 0.2469806711366302,
 'max_bins': 144,
 'interaction_cst': None,
 'validation_fraction': 0.014823091721617599,
 'n_iter_no_change': 34,
 'tol': 8.37065478390795e-08,
 'max_depth': 6}

## 100% Train

In [45]:
results = []
train_times = {}

In [46]:
name = "L_hgb_opt_01"

L_hgb_opt_01 = HistGradientBoostingClassifier(
    **fixed_params_leaf,
    **study_leaf_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_01.fit(
    x_train_hgb, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_01, x_train_hgb, y_train, x_val_hgb, y_val, train_times[name], best_iter=L_hgb_opt_01.n_iter_)
)

In [47]:
name = "L_hgb_opt_02"

L_hgb_opt_02 = HistGradientBoostingClassifier(
    **fixed_params_leaf,
    **study_leaf_no_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_02.fit(
    x_train_hgb_no_calc, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_02, x_train_hgb_no_calc, y_train, x_val_hgb_no_calc, y_val, train_times[name], best_iter=L_hgb_opt_02.n_iter_)
)

In [48]:
name = "L_hgb_opt_03"

L_hgb_opt_03 = HistGradientBoostingClassifier(
    **fixed_params_depth,
    **study_depth_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_03.fit(
    x_train_hgb, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_03, x_train_hgb, y_train, x_val_hgb, y_val, train_times[name], best_iter=L_hgb_opt_03.n_iter_)
)

In [49]:
name = "L_hgb_opt_04"

L_hgb_opt_04 = HistGradientBoostingClassifier(
    **fixed_params_depth,
    **study_depth_no_calc_loaded.best_params
)

start = time.time()
L_hgb_opt_04.fit(
    x_train_hgb_no_calc, y_train
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_hgb_opt_04, x_train_hgb_no_calc, y_train, x_val_hgb_no_calc, y_val, train_times[name], best_iter=L_hgb_opt_04.n_iter_)
)

In [50]:
pd.DataFrame(results)

,model_idx,auc_train,auc_val,gini,delta_auc,best_iter,trainingszeit
0,L_hgb_opt_01,0.691162,0.636600,0.273201,0.054562,669,32.303958
1,L_hgb_opt_02,0.679824,0.638318,0.276637,0.041506,2469,85.016068
2,L_hgb_opt_03,0.683218,0.634704,0.269407,0.048514,126,5.613486
3,L_hgb_opt_04,0.696819,0.634533,0.269067,0.062285,365,11.549956


## Save Models

In [51]:
Modelle = [
    (L_hgb_opt_01, "L_hgb_opt_01"),
    (L_hgb_opt_02, "L_hgb_opt_02"),
    (L_hgb_opt_03, "L_hgb_opt_03"),
    (L_hgb_opt_04, "L_hgb_opt_04")
]

for modell, name in Modelle:
    joblib.dump(modell, f"{name}.joblib")

joblib.dump(targ_enc, "L_hgb_targ_enc.joblib")

['L_hgb_targ_enc.joblib']

## Notizen
- Ich habe bis jetzt noch nichts zum pruning für HGB gefunden: Mines Verständnisses nach ist die Pruner implementation wie XGBPruner eine Schnittstelle durch die während des trainings gepruned werden kann. Ich setze den Median Pruner trotzdem, aber mMn wird dieser einfach keinen Effekt haben.
- J_D hatte erwähnt entweder max_depth oder max_leaf_nodes zu setzen, ich behandle es wie plain und ordered als händische versuchsreihe
- ich muss über joblib speichern
- da jd etwas anders als ich target encoded speichere ich den target encoder von hier auch mal

## Ressourcen [Abrufdatum: 19.08.2026]:
- https://www.kaggle.com/code/adrienriaux/histgradientboosting-with-optuna
- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html
- https://sklearngeneticopt.rodrigo-arenas.com/versions/latest/tutorials/tune-gradient-boosting
- https://www.cosmiclearn.com/scikitlearn/joblib.php
- https://www.geeksforgeeks.org/machine-learning/saving-a-machine-learning-model/
- https://www.simplified.guide/scikit-learn/model-persist-joblib